# DPGExplainer Saga Benchmarks — Episode 4: Wheat Seeds

A practitioner-friendly walkthrough of Decision Predicate Graphs (DPG) using the Wheat Seeds dataset.

This notebook follows the same pipeline established in Episodes 1–3:
- Train a baseline Random Forest,
- Extract DPG,
- Inspect LRC, BC, and communities,
- Compare DPG with RF importance and TreeSHAP,
- Validate class boundaries against empirical feature ranges.

**What is new in Episode 4:** a dedicated section on *configuration sensitivity* —
how DPG structure changes when you vary `decimal_threshold`, number of trees (`n_estimators`),
and tree depth (`max_depth`). This gives practitioners a principled basis for tuning DPG
rather than guessing defaults.

**Why Wheat Seeds?**  
Seven geometric grain measurements (area, perimeter, compactness, kernel length,
kernel width, asymmetry coefficient, groove length) with three wheat variety classes.
The features are physically interpretable — a threshold on `area` means distinguishing
grains by actual size — which makes threshold sensitivity discussions concrete and readable.
Complexity sits between Iris and Wine: rich enough to produce non-trivial graph structure,
simple enough that changes under configuration remain visible.

## 1. What is Explainable AI (XAI)

Explainable AI (XAI) focuses on making model behavior understandable to people.
It helps answer questions like: why was this prediction made, what features mattered most,
and whether the model behaves as intended.

Common motivations for XAI include:
- **Explain to justify:** provide evidence for decisions in high-stakes contexts.
- **Explain to discover:** surface patterns, biases, or unexpected model behaviors.
- **Explain to improve:** guide feature engineering and model selection.

For this saga, the focus is **global explanation** for tree ensembles:
not only *which* features matter, but *how* threshold predicates combine
into model-wide decision logic.

## 2. Why DPG (in one minute)

Tree ensembles, such as Random Forest, can be accurate but hard to interpret globally.
DPG converts the ensemble into a graph where:
- Nodes are predicates like `area <= 14.5` or `compactness > 0.88`.
- Edges capture how often training samples traverse those predicates in sequence.
- Metrics quantify how predicates structure the model's global reasoning.

This gives a global map of decision logic — not individual tree paths,
not feature rankings, but a connected view of *which rules route samples where*.

Two key metrics:
- **LRC (Local Reaching Centrality):** high-LRC predicates are structurally upstream;
  they shape many downstream decisions.
- **BC (Betweenness Centrality):** high-BC predicates are bridge rules;
  they connect major decision regions and sit on many routing paths.

Compared with Episodes 1–3, Wheat Seeds adds a concrete question:
*how stable are these metrics when we change how finely thresholds are rounded,
or how large the forest is?* That question matters whenever you apply DPG to a new dataset
and need to choose a configuration.

## 3. Setup (Wheat Seeds + Random Forest + DPG)

In [ ]:
%pip install --force-reinstall --no-deps git+https://github.com/Meta-Group/DPG.git

In [ ]:
import warnings
import sys
import importlib
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Force local package import so notebook uses current repo code, not a previously installed wheel.
cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'dpg' / '__init__.py').exists():
        repo_root = candidate
        break
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    for mod in list(sys.modules.keys()):
        if mod == 'dpg' or mod.startswith('dpg.'):
            importlib.reload(sys.modules[mod])

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from dpg import DPGExplainer

IMG_DIR = Path('images')
IMG_DIR.mkdir(parents=True, exist_ok=True)

# --- Load Wheat Seeds ---
# UCI Seeds dataset: 210 samples, 7 geometric grain features, 3 wheat variety classes.
# Classes: 1=Kama, 2=Rosa, 3=Canadian
WHEAT_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt"

col_names = [
    'area', 'perimeter', 'compactness',
    'kernel_length', 'kernel_width',
    'asymmetry_coeff', 'groove_length', 'variety'
]

raw = pd.read_csv(WHEAT_URL, sep=r'\s+', header=None, names=col_names)
raw['variety'] = raw['variety'].astype(int)

feature_cols = col_names[:-1]
target_names = ['Kama', 'Rosa', 'Canadian']

X = raw[feature_cols].copy()
y_raw = raw['variety'].values  # 1, 2, 3
y = y_raw - 1                  # remap to 0, 1, 2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Baseline model — same compact setup used throughout the saga
model = RandomForestClassifier(n_estimators=10, random_state=27)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=target_names))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=target_names
).plot(ax=ax, colorbar=False)
ax.set_title('Random Forest — Wheat Seeds')
fig.tight_layout()
fig.savefig(IMG_DIR / 'rf_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

### 3.1 Pair Plot (All Features by Variety)

Seven features, all geometric measurements of wheat kernels.
Before reading any graph metric, a pairplot establishes what class separation
looks like in raw feature space.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

pair_df = X.copy()
pair_df['variety'] = [target_names[i] for i in y]

g = sns.pairplot(
    pair_df,
    hue='variety',
    diag_kind='kde',
    corner=True,
    plot_kws={'alpha': 0.6, 's': 28, 'edgecolor': 'none'},
)
g.fig.suptitle('Wheat Seeds — pair plot by variety', y=1.02)
g.fig.savefig(IMG_DIR / 'pairplot.png', dpi=200, bbox_inches='tight')
plt.show()

Reading the pairplot:
- **Canadian** is largely separable from Kama and Rosa across most feature pairs —
  its kernels are smaller (lower area, perimeter, kernel length).
- **Kama and Rosa** overlap more. The cleaner separators appear in `compactness`,
  `kernel_width`, and `groove_length`.
- `asymmetry_coeff` shows broader overlap for all three varieties, suggesting it
  will play a secondary routing role in the DPG.

These patterns will be cross-checked against LRC and BC: we expect high-LRC predicates
on `area` or `kernel_length` (primary Canadian separation) and high-BC predicates
in the Kama/Rosa overlap zone.

## 4. Extracting DPG from RF

We extract the DPG from the fitted Random Forest.
`decimal_threshold=2` is the reference configuration used throughout the saga:
thresholds are rounded to two decimal places, which keeps predicates readable
without merging distinct split values.

The sensitivity experiment in Section 10 will systematically vary this parameter.

In [ ]:
explainer = DPGExplainer(
    model=model,
    feature_names=X.columns,
    target_names=target_names,
    dpg_config={
        "dpg": {
            "default": {
                "perc_var": 1e-9,
                "decimal_threshold": 2,
                "n_jobs": 1
            }
        }
    },
)

explanation = explainer.explain_global(
    X.values,
    communities=True,
    community_threshold=0.2,
)

## 5. Read the DPG Metrics

In [ ]:
explanation.node_metrics.head(10)

**Local Reaching Centrality (LRC)**
- High LRC nodes can reach many other nodes downstream.
- These predicates often act early, framing large portions of the model's logic.

**Betweenness Centrality (BC)**
- High BC nodes lie on many shortest paths between other nodes.
- These predicates are structural bottlenecks that connect major decision flows.

## 6. Compare Top LRC Predicates vs Random Forest Importance

RF importance and LRC answer different questions:
- **RF importance:** which features reduce impurity most across splits (feature-level).
- **LRC:** which concrete predicates are structurally upstream routers (predicate-level).

In Wheat Seeds, a feature like `kernel_length` may rank high in RF importance
because it is used frequently across many trees, while a specific threshold like
`kernel_length <= 5.55` may dominate LRC because that particular cut organizes
the broadest downstream routing.

In [ ]:
import dpg.visualizer as dpg_visualizer

plot_lrc_vs_rf_importance = dpg_visualizer.plot_lrc_vs_rf_importance
plot_top_lrc_predicate_splits = dpg_visualizer.plot_top_lrc_predicate_splits

plot_lrc_vs_rf_importance(
    explanation=explanation,
    model=model,
    X_df=X,
    top_k=10,
    dataset_name='Wheat Seeds',
    save_path=str(IMG_DIR / 'lrc_vs_rf_importance.png'),
    show=True,
)

plot_top_lrc_predicate_splits(
    explanation=explanation,
    X_df=X,
    y=y,
    top_predicates=5,
    top_features=2,
    dataset_name='Wheat Seeds',
    class_names=target_names,
    save_path=str(IMG_DIR / 'top_lrc_predicate_splits.png'),
    show=True,
)

### Optional: inspect top-10 LRC and RF tables

In [ ]:
top_lrc = (
    explanation.node_metrics
    .loc[
        explanation.node_metrics['Label'].astype(str).str.contains('<=', regex=False)
        | explanation.node_metrics['Label'].astype(str).str.contains('>', regex=False),
        ['Label', 'Local reaching centrality'],
    ]
    .sort_values('Local reaching centrality', ascending=False)
    .head(10)
)

top_rf = (
    pd.DataFrame({
        'feature': list(getattr(model, 'feature_names_in_', X.columns)),
        'rf_importance': model.feature_importances_,
    })
    .sort_values('rf_importance', ascending=False)
    .head(10)
)

print('=== Top-10 LRC predicates ===')
print(top_lrc.to_string(index=False))
print()
print('=== Top-10 RF features ===')
print(top_rf.to_string(index=False))

Interpretation guide:
- If a predicate has **high LRC**, it likely sets an early rule that shapes many later decisions.
- If a feature has **high RF importance**, it contributes strongly to split quality across the forest.
- Compare overlap: when high-LRC predicates and high-RF features agree, the global graph
  and model-level importance tell a consistent story.
- Divergences are informative: a feature with modest RF importance but a high-LRC predicate
  is structurally pivotal even if its impurity reduction score is not the highest.

## 7. Compare Top LRC Predicates vs TreeSHAP (Same RF Model)

TreeSHAP adds additive, per-sample feature attributions on top of the same fitted Random Forest.
This complements LRC: TreeSHAP is feature-centric and localizable, while LRC
is predicate-centric and structural.

The Wheat Seeds domain makes this comparison vivid:
TreeSHAP tells you *how much* each measurement contributed to a prediction;
LRC tells you *which threshold* in that measurement is the critical routing gate.

In [ ]:
import matplotlib.pyplot as plt
import re

try:
    import shap
except ImportError:
    shap = None

predicate_lrc = explanation.node_metrics[['Label', 'Local reaching centrality']].copy()
predicate_lrc = predicate_lrc.loc[
    predicate_lrc['Label'].astype(str).str.contains('<=', regex=False)
    | predicate_lrc['Label'].astype(str).str.contains('>', regex=False)
].copy()
predicate_lrc['feature'] = predicate_lrc['Label'].apply(
    lambda s: re.match(r'(.+?)\s*(<=|>)', str(s)).group(1).strip()
    if re.match(r'(.+?)\s*(<=|>)', str(s)) else str(s)
)
lrc_by_feature = (
    predicate_lrc
    .groupby('feature')['Local reaching centrality']
    .max()
    .reset_index()
    .sort_values('Local reaching centrality', ascending=False)
    .head(10)
)

if shap is None:
    print('TreeSHAP skipped: install with `%pip install shap`.')
    print('Showing LRC-by-feature comparison only.')

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(lrc_by_feature['feature'][::-1], lrc_by_feature['Local reaching centrality'][::-1],
            color='steelblue')
    ax.set_xlabel('Max LRC (feature-aggregated)')
    ax.set_title('LRC by feature — Wheat Seeds')
    fig.tight_layout()
    fig.savefig(IMG_DIR / 'lrc_vs_treeshap_importance.png', dpi=200, bbox_inches='tight')
    plt.show()

else:
    explainer_shap = shap.TreeExplainer(model)
    raw_shap_values = explainer_shap.shap_values(X_test)
    if isinstance(raw_shap_values, list):
        shap_values_by_class = [np.asarray(sv) for sv in raw_shap_values]
    else:
        shap_values_arr = np.asarray(raw_shap_values)
        if shap_values_arr.ndim == 2:
            shap_values_by_class = [shap_values_arr]
        elif shap_values_arr.ndim == 3:
            if shap_values_arr.shape[1] == X_test.shape[1]:
                # SHAP may return (n_samples, n_features, n_classes).
                shap_values_by_class = [
                    shap_values_arr[:, :, cls] for cls in range(shap_values_arr.shape[2])
                ]
            elif shap_values_arr.shape[2] == X_test.shape[1]:
                # Older APIs may return (n_samples, n_classes, n_features).
                shap_values_by_class = [
                    shap_values_arr[:, cls, :] for cls in range(shap_values_arr.shape[1])
                ]
            else:
                raise ValueError(
                    f'Unexpected SHAP output shape for X_test={X_test.shape}: {shap_values_arr.shape}'
                )
        else:
            raise ValueError(f'Unexpected SHAP output ndim: {shap_values_arr.ndim}')

    mean_abs_shap = np.mean(
        [np.abs(sv).mean(axis=0) for sv in shap_values_by_class], axis=0
    )
    shap_df = pd.DataFrame({
        'feature': X.columns,
        'mean_abs_shap': mean_abs_shap
    }).sort_values('mean_abs_shap', ascending=False).head(10)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].barh(lrc_by_feature['feature'][::-1],
                 lrc_by_feature['Local reaching centrality'][::-1], color='steelblue')
    axes[0].set_xlabel('Max LRC (feature-aggregated)')
    axes[0].set_title('LRC by feature')

    axes[1].barh(shap_df['feature'][::-1], shap_df['mean_abs_shap'][::-1], color='coral')
    axes[1].set_xlabel('Mean |SHAP value|')
    axes[1].set_title('TreeSHAP mean |value|')

    fig.suptitle('LRC vs TreeSHAP — Wheat Seeds', fontsize=13)
    fig.tight_layout()
    fig.savefig(IMG_DIR / 'lrc_vs_treeshap_importance.png', dpi=200, bbox_inches='tight')
    plt.show()

    print('=== Top-10 TreeSHAP features ===')
    print(shap_df.to_string(index=False))

### Optional: local TreeSHAP perspective on a single sample

In [ ]:
if shap is not None:
    sample_idx = X_test.index[0]
    sample = X_test.loc[sample_idx]
    pred_class = int(model.predict(X_test.loc[[sample_idx]])[0])
    pred_name = target_names[pred_class]

    shap_class_idx = pred_class if pred_class < len(shap_values_by_class) else 0
    sv = shap_values_by_class[shap_class_idx][X_test.index.get_loc(sample_idx)]

    print(f'Sample {sample_idx} — predicted: {pred_name}')
    sample_shap = pd.DataFrame({'feature': X.columns, 'shap': sv})
    sample_shap = sample_shap.reindex(sample_shap['shap'].abs().sort_values(ascending=False).index)
    print(sample_shap.to_string(index=False))

    # Active high-LRC predicates for this sample
    top_lrc_preds = top_lrc['Label'].tolist()
    print(f'\nTop-LRC predicates active for this sample:')
    for pred in top_lrc_preds:
        m = re.match(r'(.+?)\s*(<=|>)\s*([\-+]?[\d.]+)', str(pred))
        if not m:
            continue
        feat, op, thr = m.group(1).strip(), m.group(2), float(m.group(3))
        if feat not in sample.index:
            continue
        val = sample[feat]
        active = (val <= thr) if op == '<=' else (val > thr)
        if active:
            print(f'  ACTIVE  {pred}  (sample value: {val:.3f})')

Interpretation guide (LRC vs TreeSHAP):
- **Where TreeSHAP is better:** local explanations (per-grain/per-sample),
  signed direction of contribution, additive decomposition of model output.
- **Where LRC is better:** explicit threshold rules (`feature <= t` / `> t`),
  graph-routing influence, bottleneck detection, structural reason behind class separation.
- Together, they form a global+local view: LRC tells you which thresholds organize
  the model's logic; TreeSHAP tells you how much each measurement pushed a particular prediction.

## 8. Show BC Bottleneck Cloud in PCA Space

Betweenness centrality (BC) highlights bridge predicates between decision regions.
In Wheat Seeds, we expect BC to concentrate in the Kama/Rosa overlap zone —
these are the samples that require more complex routing before an assignment is made.

Important: this plot does **not** compute a graph BC directly on samples.
Instead, each sample receives a **BC-derived weight** from the top-BC predicates it satisfies:

`weight(sample_i) = sum_p 1[sample_i satisfies predicate_p] * BC(predicate_p)`

So a larger point means that the sample activates more of the graph predicates that behave as structural bottlenecks.

In [ ]:
import importlib
import dpg.visualizer as dpg_visualizer

dpg_visualizer = importlib.reload(dpg_visualizer)
sample_bc_weights = dpg_visualizer.sample_bc_weights
plot_sample_using_bc_weights = dpg_visualizer.plot_sample_using_bc_weights

# Each sample is scored by summing the BC of the top bottleneck predicates it satisfies.
bc_w = sample_bc_weights(explanation=explanation, X_df=X, top_k=10)
print(bc_w.describe())

_ = plot_sample_using_bc_weights(
    explanation=explanation,
    X_df=X,
    y=y,
    top_k=10,
    dataset_name='Wheat Seeds',
    class_names=target_names,
    save_path=str(IMG_DIR / 'bc_bottleneck_pca_cloud.png'),
)


## 9. Communities (Decision Themes + Class Complexity)

Community analysis summarizes how predicates organize into coherent decision modules.
Each community groups predicates that tend to co-occur along tree paths,
forming a thematic subgraph.

Beyond visualization, we quantify class complexity with class-vs-feature predicate counts:
- **Heatmap (absolute counts):** how many predicates involving each feature reach each class node.
- **Heatmap (row-normalized):** per-class feature distribution.
- **Bar plots:** predicate volume and unique feature coverage per class.

In [ ]:
import seaborn as sns
from dpg.visualizer import class_feature_predicate_counts


def plot_class_feature_complexity(heat_df, dataset_name='Wheat Seeds', top_n_features=10, save_prefix=None):
    if heat_df.empty:
        print(f'{dataset_name}: no class-feature predicate counts available.')
        return

    h = heat_df.copy()
    if h.shape[0] > top_n_features:
        h = h.loc[h.sum(axis=1).sort_values(ascending=False).index[:top_n_features]]

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(h) * 0.55 + 1)))

    sns.heatmap(
        h, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0],
        linewidths=0.3, linecolor='white'
    )
    axes[0].set_title(f'{dataset_name} — predicate counts\n(feature × class)')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Feature')

    h_norm = h.div(h.sum(axis=0) + 1e-12, axis=1)
    sns.heatmap(
        h_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
        linewidths=0.3, linecolor='white', vmin=0, vmax=1
    )
    axes[1].set_title(f'{dataset_name} — predicate distribution\n(row-normalized)')
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel('')

    fig.tight_layout()
    if save_prefix:
        fig.savefig(f'{save_prefix}_heatmap.png', dpi=200, bbox_inches='tight')
    plt.show()

    # Bar plot: total predicate count and unique feature coverage per class
    total_counts = h.sum(axis=0)
    feat_coverage = (h > 0).sum(axis=0)

    fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))
    total_counts.plot(kind='bar', ax=axes2[0], color='steelblue', edgecolor='white')
    axes2[0].set_title('Total predicate count per class')
    axes2[0].set_xlabel('Class')
    axes2[0].set_ylabel('Count')
    axes2[0].tick_params(axis='x', rotation=30)

    feat_coverage.plot(kind='bar', ax=axes2[1], color='coral', edgecolor='white')
    axes2[1].set_title('Unique feature coverage per class')
    axes2[1].set_xlabel('Class')
    axes2[1].set_ylabel('# features')
    axes2[1].tick_params(axis='x', rotation=30)

    fig2.suptitle(f'{dataset_name} — class complexity', fontsize=12)
    fig2.tight_layout()
    if save_prefix:
        fig2.savefig(f'{save_prefix}_bars.png', dpi=200, bbox_inches='tight')
    plt.show()


explanation.communities.keys()

In [ ]:
run_name = 'wheat_seeds_dpg'
explainer.plot(run_name, explanation, save_dir=str(IMG_DIR), class_flag=True, export_pdf=True, dpi=600)

In [ ]:
run_name = 'wheat_seeds_dpg'
explainer.plot_communities(run_name, explanation, save_dir=str(IMG_DIR), class_flag=True, export_pdf=True, dpi=600)

### 9.1 Communities and predicates per community

The community-colored DPG plot shows the global modules. The next cell makes those modules explicit by listing, for each community, its class association, predicate budget, dominant features, and the top predicates by LRC.

In [ ]:
from collections import Counter


def community_predicate_study(explanation, top_k=8):
    node_df = explanation.node_metrics.copy()
    predicate_mask = (
        node_df['Label'].astype(str).str.contains('<=', regex=False)
        | node_df['Label'].astype(str).str.contains('>', regex=False)
    )
    predicate_df = node_df.loc[predicate_mask].copy()

    label_to_rows = {}
    node_to_rows = {}
    for idx, row in predicate_df.iterrows():
        label_to_rows.setdefault(str(row['Label']), []).append(idx)
        node_to_rows.setdefault(str(row['Node']), []).append(idx)

    communities_obj = explanation.communities or {}
    if isinstance(communities_obj, dict) and 'Clusters' in communities_obj:
        raw_specs = list(communities_obj['Clusters'].items())
    elif isinstance(communities_obj, dict) and 'Communities' in communities_obj:
        raw_specs = [(f'Community {i}', members) for i, members in enumerate(communities_obj['Communities'])]
    else:
        raw_specs = []

    rows = []
    for community_id, (raw_name, members) in enumerate(raw_specs):
        class_name = str(raw_name).replace('Class ', '')
        if class_name.lower() == 'ambiguous':
            class_name = 'Ambiguous'

        matched = []
        seen = set()
        for member in members:
            member_key = str(member)
            for idx in node_to_rows.get(member_key, []):
                if idx not in seen:
                    matched.append(idx)
                    seen.add(idx)
            for idx in label_to_rows.get(member_key, []):
                if idx not in seen:
                    matched.append(idx)
                    seen.add(idx)

        if not matched:
            continue

        community_df = predicate_df.loc[matched].copy()
        community_df['community_id'] = community_id
        community_df['class_name'] = class_name
        community_df['feature'] = community_df['Label'].apply(
            lambda s: re.match(r'(.+?)\\s*(<=|>)', str(s)).group(1).strip()
            if re.match(r'(.+?)\\s*(<=|>)', str(s)) else str(s)
        )
        community_df['operator'] = community_df['Label'].apply(
            lambda s: re.match(r'(.+?)\\s*(<=|>)', str(s)).group(2)
            if re.match(r'(.+?)\\s*(<=|>)', str(s)) else ''
        )
        community_df['threshold'] = community_df['Label'].apply(
            lambda s: float(re.match(r'(.+?)\\s*(<=|>)\\s*([\\-+]?[\\d.]+)', str(s)).group(3))
            if re.match(r'(.+?)\\s*(<=|>)\\s*([\\-+]?[\\d.]+)', str(s)) else np.nan
        )
        rows.append(community_df)

    if not rows:
        print('No community predicates found.')
        return pd.DataFrame(), pd.DataFrame()

    community_predicates = pd.concat(rows, ignore_index=True)
    summary_rows = []
    for (community_id, class_name), grp in community_predicates.groupby(['community_id', 'class_name'], sort=True):
        feature_counts = Counter(grp['feature'])
        dominant_features = ', '.join(f'{feat} ({cnt})' for feat, cnt in feature_counts.most_common(3))
        summary_rows.append({
            'community_id': community_id,
            'class_name': class_name,
            'n_predicates': int(len(grp)),
            'n_features': int(grp['feature'].nunique()),
            'dominant_features': dominant_features,
        })

    summary_df = pd.DataFrame(summary_rows).sort_values(['class_name', 'n_predicates'], ascending=[True, False])
    print('=== Community summary ===')
    print(summary_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.55 * len(summary_df) + 1)))
    plot_df = summary_df.copy()
    plot_df['community_label'] = plot_df.apply(
        lambda r: f"C{int(r['community_id'])} ({r['class_name']})", axis=1
    )
    colors = {
        'Kama': '#1f77b4',
        'Rosa': '#ff7f0e',
        'Canadian': '#2ca02c',
        'Ambiguous': '#7f7f7f',
    }
    ax.barh(
        plot_df['community_label'][::-1],
        plot_df['n_predicates'][::-1],
        color=[colors.get(c, '#7f7f7f') for c in plot_df['class_name'][::-1]],
    )
    ax.set_xlabel('Predicate count')
    ax.set_title('Wheat Seeds — predicate budget per community')
    fig.tight_layout()
    fig.savefig(IMG_DIR / 'community_predicate_budget.png', dpi=200, bbox_inches='tight')
    plt.show()

    print('\n=== Top predicates per community (by LRC) ===')
    for _, summary_row in summary_df.iterrows():
        cid = summary_row['community_id']
        cname = summary_row['class_name']
        print(f'\nCommunity {cid} ({cname})')
        cols = ['Label', 'feature', 'Local reaching centrality', 'Betweenness centrality']
        top_df = (
            community_predicates[
                (community_predicates['community_id'] == cid)
                & (community_predicates['class_name'] == cname)
            ][cols]
            .sort_values('Local reaching centrality', ascending=False)
            .head(top_k)
        )
        print(top_df.to_string(index=False))

    return summary_df, community_predicates


community_summary, community_predicates = community_predicate_study(explanation, top_k=8)

In [ ]:
heat = class_feature_predicate_counts(explanation)
plot_class_feature_complexity(
    heat,
    dataset_name='Wheat Seeds',
    top_n_features=10,
    save_prefix=str(IMG_DIR / 'communities_class_feature_complexity')
)

## 9.2 DPG Community Ranges vs Dataset Ranges

In [ ]:
from dpg.visualizer import (
    classwise_feature_bounds_from_communities,
    class_feature_predicate_positions,
    class_lookup_from_target_names,
    plot_dpg_class_bounds_vs_dataset_feature_ranges,
)

class_bounds = classwise_feature_bounds_from_communities(explanation)
class_lookup = class_lookup_from_target_names(target_names)
predicate_positions = class_feature_predicate_positions(explanation)

_ = plot_dpg_class_bounds_vs_dataset_feature_ranges(
    explanation=explanation,
    X_df=X,
    y=y,
    dataset_name='Wheat Seeds',
    top_features=7,
    feature_cols_per_row=4,
    class_lookup=class_lookup,
    predicate_positions=predicate_positions,
    class_bounds=class_bounds,
    save_path=str(IMG_DIR / 'dpg_vs_dataset_feature_ranges.png'),
)

How to read this figure:
- **Gray thick bars:** empirical dataset class ranges for each feature.
- **Blue bars:** DPG community-derived ranges.
- **Triangle markers at edges:** unbounded sides (`-inf` / `+inf`) from one-sided predicates.
- **Green `^` / red `v` triangles:** predicate-threshold density by operator (`>` / `<=`);
  dense clusters indicate where the forest repeatedly concentrates decision logic.

In Wheat Seeds, expect tight DPG bounds on `area` and `kernel_length` for Canadian
(its smaller size is well-bounded), and wider or asymmetric bounds for Kama/Rosa
where overlap forces the model to rely on combinations of weaker rules.

---

## 10. Configuration Sensitivity Experiments

The previous sections established the reference DPG (`decimal_threshold=2`, `n_estimators=10`,
no depth constraint). This section systematically varies each parameter to show
how DPG structure responds.

Three experiments:
1. **Threshold resolution** (`decimal_threshold` ∈ {1, 2, 3, 4}) — how rounding changes predicate count and LRC rankings.
2. **Forest size** (`n_estimators` ∈ {5, 10, 30, 50}) — how graph density scales with ensemble size.
3. **Tree depth** (`max_depth` ∈ {3, 5, 7, None}) — how depth controls predicate coverage and BC distribution.

The goal is not to tune a better model, but to understand what DPG signals are stable
across configurations and what is sensitive to tuning choices.

### 10.1 Helper — Build DPG Snapshot

We define a helper that fits a new model and DPG under a given configuration
and returns a compact summary: predicate count, edge count, community count,
and the top-5 LRC predicate labels.

In [ ]:
def build_dpg_snapshot(
    X, y,
    n_estimators=10,
    max_depth=None,
    decimal_threshold=2,
    random_state=27,
):
    """
    Train a Random Forest and extract a DPG snapshot under the given configuration.

    Returns a dict with graph-level statistics and the top-5 LRC predicates.
    """
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
    )
    rf.fit(X_train, y_train)

    exp_local = DPGExplainer(
        model=rf,
        feature_names=X.columns,
        target_names=target_names,
        dpg_config={
            'dpg': {
                'default': {
                    'perc_var': 1e-9,
                    'decimal_threshold': decimal_threshold,
                    'n_jobs': 1,
                }
            }
        },
    )
    expl = exp_local.explain_global(
        X.values,
        communities=True,
        community_threshold=0.2,
    )

    nm = expl.node_metrics
    predicate_nodes = nm[
        nm['Label'].astype(str).str.contains('<=', regex=False)
        | nm['Label'].astype(str).str.contains('>', regex=False)
    ]
    n_predicates = len(predicate_nodes)
    n_edges = expl.edge_metrics.shape[0] if expl.edge_metrics is not None else 0

    clusters = expl.communities.get('Clusters', expl.communities.get('Communities', [])) \
        if expl.communities else []
    n_communities = len(clusters) if clusters else 0

    top5_lrc = (
        predicate_nodes
        .sort_values('Local reaching centrality', ascending=False)
        .head(5)['Label']
        .tolist()
    )

    return {
        'n_predicates': n_predicates,
        'n_edges': n_edges,
        'n_communities': n_communities,
        'top5_lrc': top5_lrc,
        'node_metrics': nm,
    }

print('Helper ready.')

### 10.2 Experiment 1 — Effect of `decimal_threshold`

`decimal_threshold` controls how many decimal places are kept when rounding split values.

- **Coarser rounding (dt=1):** nearby thresholds collapse into the same predicate node.
  The graph is smaller and simpler, but some threshold distinctions are lost.
- **Finer rounding (dt=4):** more distinct nodes, closer to the raw tree splits.
  The graph is larger and more granular, but visually harder to navigate.

The key question is whether *LRC rankings remain stable* under rounding variation.
If the top features stay the same across dt=1 to dt=4, the explanation is robust.
If rankings shift substantially, the choice of `decimal_threshold` is load-bearing.

In [ ]:
dt_values = [1, 2, 3, 4]
dt_results = {}

for dt in dt_values:
    snap = build_dpg_snapshot(X, y, n_estimators=10, max_depth=None, decimal_threshold=dt)
    dt_results[dt] = snap
    print(f'dt={dt}: {snap["n_predicates"]} predicates, '
          f'{snap["n_edges"]} edges, '
          f'{snap["n_communities"]} communities')

In [ ]:
# --- Graph size summary plot ---
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

metrics_keys = ['n_predicates', 'n_edges', 'n_communities']
titles = ['Predicate nodes', 'Edges', 'Communities']
colors = ['steelblue', 'seagreen', 'coral']

for ax, key, title, color in zip(axes, metrics_keys, titles, colors):
    vals = [dt_results[dt][key] for dt in dt_values]
    ax.bar([str(dt) for dt in dt_values], vals, color=color, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('decimal_threshold')
    ax.set_ylabel('Count')
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals) * 0.02, str(v), ha='center', fontsize=10)

fig.suptitle('DPG graph size vs decimal_threshold — Wheat Seeds', fontsize=12)
fig.tight_layout()
fig.savefig(IMG_DIR / 'sensitivity_dt_graph_size.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# --- LRC ranking stability across decimal_threshold values ---

def top_lrc_features(node_metrics, top_k=5):
    """Return top-k feature names ranked by max LRC across their predicates."""
    nm = node_metrics[
        node_metrics['Label'].astype(str).str.contains('<=', regex=False)
        | node_metrics['Label'].astype(str).str.contains('>', regex=False)
    ].copy()
    nm['feature'] = nm['Label'].apply(
        lambda s: re.match(r'(.+?)\s*(<=|>)', str(s)).group(1).strip()
        if re.match(r'(.+?)\s*(<=|>)', str(s)) else str(s)
    )
    ranked = (
        nm.groupby('feature')['Local reaching centrality']
        .max()
        .sort_values(ascending=False)
    )
    return ranked.head(top_k)


fig, axes = plt.subplots(1, len(dt_values), figsize=(14, 4), sharey=False)

for ax, dt in zip(axes, dt_values):
    ranked = top_lrc_features(dt_results[dt]['node_metrics'], top_k=7)
    ax.barh(ranked.index[::-1], ranked.values[::-1], color='steelblue', edgecolor='white')
    ax.set_title(f'dt = {dt}')
    ax.set_xlabel('Max LRC')
    if dt == dt_values[0]:
        ax.set_ylabel('Feature')

fig.suptitle('Top LRC features by decimal_threshold — Wheat Seeds', fontsize=12)
fig.tight_layout()
fig.savefig(IMG_DIR / 'sensitivity_dt_lrc_ranking.png', dpi=200, bbox_inches='tight')
plt.show()

# Print ranking overlap summary
ref_features = set(top_lrc_features(dt_results[2]['node_metrics'], top_k=5).index)
print('LRC top-5 feature overlap vs dt=2 (reference):')
for dt in dt_values:
    other = set(top_lrc_features(dt_results[dt]['node_metrics'], top_k=5).index)
    overlap = len(ref_features & other)
    print(f'  dt={dt}: {overlap}/5 features in common')

**Reading the sensitivity to `decimal_threshold`:**
- Predicate count grows with `dt`: finer rounding splits one merged node into multiple nodes
  for nearby thresholds.
- Community count may not grow proportionally — communities reflect structural clusters,
  not raw predicate count.
- If LRC feature rankings are stable (high overlap across dt values), the explanation
  is robust to threshold resolution. If rankings shift, the structural signal
  depends on the specific rounding choice.

**Practical guidance:** start with `decimal_threshold=2` for exploration.
Use dt=1 for a fast high-level view. Only increase to dt=3/4 when precise threshold
values matter to stakeholders (e.g., auditing a specific split value).

### 10.3 Experiment 2 — Effect of Forest Size (`n_estimators`)

Larger forests produce richer DPGs: more trees contribute more paths,
so more predicate combinations are observed.

The question is whether the *structural signals* (LRC rankings, community themes)
converge as `n_estimators` grows, or whether they keep changing.
Convergence is a sign of a stable explanation.

In [ ]:
n_est_values = [5, 10, 30, 50]
n_est_results = {}

for n in n_est_values:
    snap = build_dpg_snapshot(X, y, n_estimators=n, max_depth=None, decimal_threshold=2)
    n_est_results[n] = snap
    print(f'n_estimators={n:3d}: {snap["n_predicates"]} predicates, '
          f'{snap["n_edges"]} edges, '
          f'{snap["n_communities"]} communities')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, key, title, color in zip(axes, metrics_keys, titles, colors):
    vals = [n_est_results[n][key] for n in n_est_values]
    ax.plot([str(n) for n in n_est_values], vals, marker='o', color=color, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('n_estimators')
    ax.set_ylabel('Count')
    for i, v in enumerate(vals):
        ax.annotate(str(v), (i, v), textcoords='offset points', xytext=(0, 7),
                    ha='center', fontsize=9)

fig.suptitle('DPG graph size vs n_estimators — Wheat Seeds', fontsize=12)
fig.tight_layout()
fig.savefig(IMG_DIR / 'sensitivity_nest_graph_size.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(n_est_values), figsize=(14, 4), sharey=False)

for ax, n in zip(axes, n_est_values):
    ranked = top_lrc_features(n_est_results[n]['node_metrics'], top_k=7)
    ax.barh(ranked.index[::-1], ranked.values[::-1], color='seagreen', edgecolor='white')
    ax.set_title(f'n_est = {n}')
    ax.set_xlabel('Max LRC')
    if n == n_est_values[0]:
        ax.set_ylabel('Feature')

fig.suptitle('Top LRC features by n_estimators — Wheat Seeds', fontsize=12)
fig.tight_layout()
fig.savefig(IMG_DIR / 'sensitivity_nest_lrc_ranking.png', dpi=200, bbox_inches='tight')
plt.show()

ref_features_n = set(top_lrc_features(n_est_results[10]['node_metrics'], top_k=5).index)
print('LRC top-5 feature overlap vs n_estimators=10 (reference):')
for n in n_est_values:
    other = set(top_lrc_features(n_est_results[n]['node_metrics'], top_k=5).index)
    overlap = len(ref_features_n & other)
    print(f'  n_estimators={n:3d}: {overlap}/5 features in common')

**Reading the sensitivity to `n_estimators`:**
- Graph size typically grows with `n_estimators` up to a saturation point:
  once all relevant predicate combinations have been seen, adding more trees
  adds edge weight but few new nodes.
- If LRC rankings stabilize from n=10 to n=30, the reference configuration is adequate.
  If they still shift at n=50, a larger forest may be warranted for stable explanations.
- Community count can *decrease* as the forest grows, if more evidence consolidates
  scattered communities into stronger, fewer clusters.

**Practical guidance:** for exploratory DPG, `n_estimators=10` gives a fast pass.
For a final explanation to share with stakeholders, prefer `n_estimators≥30`
and verify that LRC rankings have converged.

### 10.4 Experiment 3 — Effect of Tree Depth (`max_depth`)

Tree depth controls how many predicate levels each sample can traverse.

- **Shallow trees (`max_depth=3`):** short paths → few predicates per path →
  sparse DPG dominated by root-level splits. BC concentrates at the first splits.
- **Deep trees (`max_depth=None`):** long paths → many predicates per path →
  dense DPG with broader coverage. BC spreads deeper into the graph.

For Wheat Seeds, shallow trees may cleanly separate Canadian (easily identified
by size features) but struggle with Kama/Rosa overlap, which needs deeper predicates.

In [ ]:
depth_values = [3, 5, 7, None]
depth_labels = ['3', '5', '7', 'None']
depth_results = {}

for depth, label in zip(depth_values, depth_labels):
    snap = build_dpg_snapshot(X, y, n_estimators=10, max_depth=depth, decimal_threshold=2)
    depth_results[label] = snap
    print(f'max_depth={label:4s}: {snap["n_predicates"]} predicates, '
          f'{snap["n_edges"]} edges, '
          f'{snap["n_communities"]} communities')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, key, title, color in zip(axes, metrics_keys, titles, colors):
    vals = [depth_results[l][key] for l in depth_labels]
    ax.bar(depth_labels, vals, color=color, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('max_depth')
    ax.set_ylabel('Count')
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals) * 0.02, str(v), ha='center', fontsize=10)

fig.suptitle('DPG graph size vs max_depth — Wheat Seeds', fontsize=12)
fig.tight_layout()
fig.savefig(IMG_DIR / 'sensitivity_depth_graph_size.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(depth_labels), figsize=(14, 4), sharey=False)

for ax, label in zip(axes, depth_labels):
    ranked = top_lrc_features(depth_results[label]['node_metrics'], top_k=7)
    ax.barh(ranked.index[::-1], ranked.values[::-1], color='coral', edgecolor='white')
    ax.set_title(f'max_depth = {label}')
    ax.set_xlabel('Max LRC')
    if label == depth_labels[0]:
        ax.set_ylabel('Feature')

fig.suptitle('Top LRC features by max_depth — Wheat Seeds', fontsize=12)
fig.tight_layout()
fig.savefig(IMG_DIR / 'sensitivity_depth_lrc_ranking.png', dpi=200, bbox_inches='tight')
plt.show()

ref_features_d = set(top_lrc_features(depth_results['None']['node_metrics'], top_k=5).index)
print('LRC top-5 feature overlap vs max_depth=None (reference):')
for label in depth_labels:
    other = set(top_lrc_features(depth_results[label]['node_metrics'], top_k=5).index)
    overlap = len(ref_features_d & other)
    print(f'  max_depth={label:4s}: {overlap}/5 features in common')

**Reading the sensitivity to `max_depth`:**
- Shallow trees produce sparse DPGs with high-BC concentration at the top-level splits.
  These are useful for a quick high-level summary but miss the fine-grained routing
  that separates closely overlapping classes.
- Deep (unconstrained) trees produce denser DPGs where overlap-resolution logic
  is visible deeper in the graph. Communities become more meaningful because
  there is enough path variety to form distinct structural clusters.
- For Wheat Seeds, the Kama/Rosa overlap requires at least moderate depth to resolve;
  with `max_depth=3`, the DPG may capture Canadian separation cleanly
  but collapse Kama/Rosa into the same community.

**Practical guidance:** avoid `max_depth=3` for DPG analysis unless a fast overview
is the goal. Unconstrained depth (`max_depth=None`) gives the richest DPG
and is the recommended default for global interpretability.

### 10.5 Sensitivity Summary

Consolidated view of how all three configuration axes affect graph size.

In [ ]:
rows = []
for dt in dt_values:
    s = dt_results[dt]
    rows.append({'axis': 'decimal_threshold', 'value': str(dt),
                 'predicates': s['n_predicates'], 'edges': s['n_edges'],
                 'communities': s['n_communities']})
for n in n_est_values:
    s = n_est_results[n]
    rows.append({'axis': 'n_estimators', 'value': str(n),
                 'predicates': s['n_predicates'], 'edges': s['n_edges'],
                 'communities': s['n_communities']})
for label in depth_labels:
    s = depth_results[label]
    rows.append({'axis': 'max_depth', 'value': label,
                 'predicates': s['n_predicates'], 'edges': s['n_edges'],
                 'communities': s['n_communities']})

summary_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axis_names = ['decimal_threshold', 'n_estimators', 'max_depth']
palette_map = {'decimal_threshold': 'steelblue', 'n_estimators': 'seagreen', 'max_depth': 'coral'}

for ax_idx, (ax, metric, title) in enumerate(
    zip(axes, ['predicates', 'edges', 'communities'], titles)
):
    for i, axis_name in enumerate(axis_names):
        sub = summary_df[summary_df['axis'] == axis_name]
        x_pos = np.arange(len(sub)) + i * (len(sub) + 1)
        ax.bar(x_pos, sub[metric].values,
               color=palette_map[axis_name],
               label=axis_name if ax_idx == 0 else '',
               edgecolor='white', alpha=0.85)
        ax.set_xticks([])

    ax.set_title(title)
    ax.set_ylabel('Count')
    ax.set_xlabel('Configuration runs')

handles = [
    plt.Rectangle((0, 0), 1, 1, color=palette_map[a], label=a)
    for a in axis_names
]
axes[0].legend(handles=handles, fontsize=8, loc='upper left')

fig.suptitle('Sensitivity summary — all three axes — Wheat Seeds', fontsize=12)
fig.tight_layout()
fig.savefig(IMG_DIR / 'sensitivity_summary.png', dpi=200, bbox_inches='tight')
plt.show()

print(summary_df.to_string(index=False))

---

## 11. Takeaway Messages

### Standard DPG pipeline (Sections 3–9)

- **Canadian is the simple class:** its smaller, more compact grain geometry is captured
  by a handful of high-LRC predicates on `area`, `kernel_length`, or `perimeter`.
  Boundaries are tight and well-aligned with empirical data ranges.
- **Kama/Rosa is where the model works harder:** high-BC predicates concentrate
  in their overlap zone (visible in the PCA cloud). Communities in this region
  are shared between the two varieties, reflecting genuine ambiguity.
- **LRC and TreeSHAP agree at the feature level** on the dominant drivers,
  while LRC adds the specific threshold semantics and structural routing role
  that TreeSHAP does not provide.

### Configuration sensitivity (Section 10)

| Parameter | Effect on graph | Effect on LRC ranking | Recommendation |
|---|---|---|---|
| `decimal_threshold` ↑ | More predicate nodes | Feature-level stable, predicate-level changes | Use dt=2 by default; dt=1 for fast overview |
| `n_estimators` ↑ | More edges, nodes saturate | Rankings converge with more trees | Use ≥30 for final explanations |
| `max_depth` ↑ | More nodes and edges | More features appear at deeper depths | Use unconstrained for full DPG richness |

One-line conclusion:  
**DPG feature-level rankings (which features drive the model) are robust to configuration.
Predicate-level specifics (which exact threshold) are sensitive to `decimal_threshold`.
Graph structure (density, community count) is sensitive to all three axes.
Tune to task: fast overview → (dt=1, n=10, depth=5); robust final explanation → (dt=2, n≥30, depth=None).**

## 12. References

### Original DPG proposal
- Arrighi, L., Pennella, L., Marques Tavares, G., Barbon Junior, S.  
  **Decision Predicate Graphs: Enhancing Interpretability in Tree Ensembles**.  
  *World Conference on Explainable Artificial Intelligence*, 311-332.  
  https://link.springer.com/chapter/10.1007/978-3-031-63797-1_16

### Extended DPG (Isolation Forest)
- Ceschin, M., Arrighi, L., Longo, L., Barbon Junior, S.  
  **Extending Decision Predicate Graphs for Comprehensive Explanation of Isolation Forest**.  
  *World Conference on Explainable Artificial Intelligence*, 271-293.  
  https://link.springer.com/chapter/10.1007/978-3-032-08324-1_12

### SHAP / TreeSHAP
- Lundberg, S. M., Lee, S.-I.  
  **A Unified Approach to Interpreting Model Predictions**.  
  *NeurIPS 2017*.  
  https://proceedings.neurips.cc/paper_files/paper/2017/hash/8a20a8621978632d76c43dfd28b67767-Abstract.html

### Wheat Seeds Dataset
- Charytanowicz, M., Niewczas, J., Kulczycki, P., Kowalski, P. A., Łukasik, S., Żak, S.  
  **Complete Gradient Clustering Algorithm for Features Analysis of X-Ray Images**.  
  *Information Technologies in Biomedicine*, 2010.  
  UCI ML Repository: https://archive.ics.uci.edu/ml/datasets/seeds

### Saga context
- Episode 1 (Iris): introduced the baseline DPG workflow and the core idea that Random Forest accuracy alone does not reveal the decision program behind predictions.  
  https://medium.com/@sbarbonjr/dpgexplainer-saga-benchmarks-episode-1-iris-c8816db2857d
- Episode 2 (Wine): kept the same workflow but moved to a harder multiclass setting with 13 continuous features, denser overlap, stronger feature interactions, and a larger rule budget.
  https://medium.com/@sbarbonjr/dpgexplainer-saga-benchmarks-episode-2-wine-b6c105ff2ca0
- Episode 3 (Breast Cancer): extended the saga to a high-dimensional binary problem, adding TreeSHAP to compare attribution-level explanations with DPG structural signals such as LRC and bottlenecks.
  https://medium.com/@sbarbonjr/dpgexplainer-saga-benchmarks-episode-1-breast-cancer-dd38765903be
- Episode 4 (Wheat Seeds): keeps the same interpretability stack while adding configuration sensitivity, showing how graph structure and key metrics change as `decimal_threshold`, `n_estimators`, and `max_depth` vary.